# Pilotage du benchmark de challengers

Ce notebook lance le module `challenger_benchmark` **etape par etape**. Chaque section est un point d'arret : execute jusqu'a la section voulue et stoppe la si tu veux inspecter avant de continuer.

**Deux modes d'usage**
- *Exploration fine* (sections 5) : un seul modele, sous-etape par sous-etape (tuning -> ajustement -> SHAP -> export), pour regarder dans le detail et t'arreter ou tu veux.
- *Execution complete* (section 6) : la boucle sur tous les modeles, avec suivi.

**Points d'arret recommandes** : apres la section 2 (valider les variables), apres la section 3 (valider les donnees et le taux de defaut), apres le tuning d'un modele (regarder `best_params` avant d'ajuster).

> A placer au meme endroit que `challenger.yaml` (racine du module).

## 0. Initialisation

In [ ]:
# Acces au package (le code est dans ./src) + rechargement a chaud pendant le dev
import sys, gc
from pathlib import Path

BASE = Path.cwd()
SRC = BASE / "src"
if not SRC.exists():
    # repli si le notebook est lance depuis un autre dossier de travail
    SRC = Path(__file__).resolve().parent / "src" if "__file__" in dir() else SRC
sys.path.insert(0, str(SRC))

%load_ext autoreload
%autoreload 2

import pandas as pd
from IPython.display import Image, display

from challenger_benchmark.config import _parse_config
from challenger_benchmark import data as data_mod
from challenger_benchmark.models import build_models
from challenger_benchmark.tuning import tune
from challenger_benchmark.evaluation import evaluate
from challenger_benchmark.explain import sample_for_shap, compute_shap, variable_importance
from challenger_benchmark import plots, export
import yaml

def show(path):
    """Affiche une figure PNG dans le notebook."""
    display(Image(filename=str(path)))

print("Imports ok. Dossier de travail :", BASE)

## 1. Configuration

Charge `challenger.yaml`. Les trois `*_OVERRIDE` permettent de tester vite sans toucher au YAML : par exemple `N_TRIALS_OVERRIDE = 5` pour un essai rapide, ou restreindre `MODELS_OVERRIDE` a un seul modele. Laisse a `None` pour utiliser le YAML tel quel.

In [ ]:
CONFIG_PATH = "challenger.yaml"

# Surcharges optionnelles (None = valeur du YAML)
N_TRIALS_OVERRIDE  = None      # ex. 5 pour un essai rapide
MODELS_OVERRIDE    = None      # ex. ["xgboost"] ou ["logistic_regression", "xgboost"]
OUTPUT_DIR_OVERRIDE = None     # ex. "outputs_test"

with open(CONFIG_PATH, "r", encoding="utf-8") as fh:
    raw = yaml.safe_load(fh)

if N_TRIALS_OVERRIDE is not None:
    raw.setdefault("tuning", {})["n_trials"] = N_TRIALS_OVERRIDE
if MODELS_OVERRIDE is not None:
    raw["models"] = MODELS_OVERRIDE
if OUTPUT_DIR_OVERRIDE is not None:
    raw["output_dir"] = OUTPUT_DIR_OVERRIDE

cfg = _parse_config(raw)

print("Base       :", cfg.data.path)
print("Cible      :", cfg.data.target)
print("Sortie     :", cfg.output_dir)
print("Modeles    :", list(cfg.models))
print("Optuna     :", cfg.tuning.n_trials, "essais,", cfg.tuning.cv_folds, "plis")
print("SHAP       :", cfg.shap.sample_size, "lignes, top", cfg.shap.top_n)

## 2. Variables et types  ·  *point d'arret*

Reconstitue la liste des variables depuis `drivers.xlsx` et separe quantitatives / qualitatives. **Arrete-toi ici pour valider** que la liste et les types sont corrects avant tout calcul lourd.

In [ ]:
drivers = data_mod.load_drivers(cfg)
variables = data_mod.resolve_variables(cfg, drivers)
num_features, cat_features = data_mod.split_feature_types(cfg, drivers, variables)

print(f"{len(variables)} variables retenues : {len(num_features)} quantitatives, {len(cat_features)} qualitatives\n")
print("Quantitatives :", num_features)
print("\nQualitatives  :", cat_features)

## 3. Chargement des donnees  ·  *point d'arret*

Scan Polars projete, decodage des modalites, split train/test via la colonne `sample`. On inspecte les volumes, le taux de defaut de chaque echantillon et les taux de manquants. **Arrete-toi ici** si un taux de defaut ou de manquant te surprend.

In [ ]:
X_train, y_train, X_test, y_test = data_mod.load_dataset(
    cfg, variables, num_features, cat_features
)

print(f"Train : {X_train.shape[0]:>7} lignes | taux de defaut = {y_train.mean():.3%}")
print(f"Test  : {X_test.shape[0]:>7} lignes | taux de defaut = {y_test.mean():.3%}")

miss = (X_train.isna().mean().sort_values(ascending=False) * 100).round(2)
print("\nTaux de manquants train (%), variables concernees :")
display(miss[miss > 0].to_frame("manquants_%"))

## 4. Modeles a lancer

Instancie les challengers de la config. Tu peux en isoler un pour l'exploration fine de la section 5.

In [ ]:
models = build_models(cfg.models, num_features, cat_features, cfg.tuning.seed)
models_by_name = {m.name: m for m in models}
descriptions = {v: (str(d) if d is not None else "")
                for v, d in zip(drivers[cfg.drivers.variable_col].to_list(),
                                drivers[cfg.drivers.description_col].to_list())
                if v in variables}

print("Modeles prets :", list(models_by_name))

## 5. Exploration fine d'un seul modele

Mode pas-a-pas : choisis un modele, puis execute les sous-etapes une par une. Tu peux t'arreter apres n'importe laquelle (par exemple regarder `best_params` avant d'ajuster, ou la courbe ROC avant de calculer le SHAP).

> Tu peux interrompre la cellule de tuning a tout moment : Optuna conserve les essais deja termines et `study.best_params` reste disponible.

In [ ]:
# --- 5.0  Choix du modele a explorer ---
MODEL_NAME = "xgboost"   # parmi : logistic_regression, hist_gradient_boosting, xgboost, catboost
model = models_by_name[MODEL_NAME]
out_folder = export.model_dir(Path(cfg.output_dir), model.name)
print("Modele selectionne :", model.name, "| dossier de sortie :", out_folder)

In [ ]:
# --- 5.1  Optimisation des hyperparametres (suivi essai par essai) ---
def trace(study, trial):
    best = study.best_value
    val = trial.value if trial.value is not None else float("nan")
    print(f"  essai {trial.number + 1:>3}/{cfg.tuning.n_trials} | "
          f"AUC = {val:.4f} | meilleur = {best:.4f}")

best_params, study = tune(model, X_train, y_train, cfg.tuning, callbacks=[trace])

print("\nMeilleur AUC (CV) :", round(study.best_value, 4))
print("Meilleurs hyperparametres :", best_params)
plots.plot_optuna_history(study, out_folder / "optuna_history.png")
show(out_folder / "optuna_history.png")

In [ ]:
# --- 5.2  Ajustement sur tout le train + evaluation sur le test ---
estimator = model.build(best_params)
model.fit(estimator, X_train, y_train)

p_train = model.predict_proba(estimator, X_train)
p_test  = model.predict_proba(estimator, X_test)
metrics = evaluate(model.name, y_train, p_train, y_test, p_test)

display(pd.DataFrame([metrics.to_row()]).set_index("model").round(4))

plots.plot_roc(y_train, p_train, y_test, p_test,
               metrics.train.auc, metrics.test.auc, out_folder / "roc.png")
show(out_folder / "roc.png")

In [ ]:
# --- 5.3  SHAP : importance et impact ---
X_shap = sample_for_shap(X_test, cfg.shap.sample_size, cfg.tuning.seed)
sv, feat_names, disp, col_to_var = compute_shap(model, estimator, X_shap)
importance = variable_importance(sv, feat_names, col_to_var)

plots.plot_shap_importance(importance, out_folder / "shap_importance.png",
                           cfg.shap.top_n, descriptions)
plots.plot_shap_beeswarm(sv, disp, feat_names, out_folder / "shap_impact.png", cfg.shap.top_n)
show(out_folder / "shap_importance.png")
show(out_folder / "shap_impact.png")

In [ ]:
# --- 5.4  Export des artefacts de ce modele ---
export.save_model(estimator, best_params, importance, out_folder)
print("Exporte dans :", out_folder)
for f in sorted(out_folder.iterdir()):
    print("  -", f.name)

# Liberation memoire avant de passer a un autre modele
del estimator, study, p_train, p_test, sv, disp, X_shap
gc.collect()

## 6. Execution complete (tous les modeles)

Deux variantes au choix.

**6a — Boucle suivie** : meme sequence que la section 5, deroulee sur tous les modeles, avec affichage des figures au fil de l'eau et liberation memoire entre modeles. Tu vois la progression et peux interrompre proprement.

**6b — Une seule ligne** : delegue tout au module (`run_with_config`), sans affichage intermediaire. A utiliser quand tu as confiance et veux juste les sorties.

In [ ]:
# --- 6a  Boucle suivie sur tous les modeles selectionnes ---
output_dir = Path(cfg.output_dir)
output_dir.mkdir(parents=True, exist_ok=True)
summary_rows, importances = [], {}

for m in models:
    print(f"\n=== {m.name} ===")
    bp, st = tune(m, X_train, y_train, cfg.tuning)
    est = m.build(bp); m.fit(est, X_train, y_train)
    ptr, pte = m.predict_proba(est, X_train), m.predict_proba(est, X_test)
    met = evaluate(m.name, y_train, ptr, y_test, pte)
    summary_rows.append(met.to_row())
    print(f"AUC test = {met.test.auc:.4f} | Gini test = {met.test.gini:.4f} | ecart = {met.auc_gap:+.4f}")

    folder = export.model_dir(output_dir, m.name)
    Xs = sample_for_shap(X_test, cfg.shap.sample_size, cfg.tuning.seed)
    s, fn, dp, c2v = compute_shap(m, est, Xs)
    imp = variable_importance(s, fn, c2v); importances[m.name] = imp

    plots.plot_roc(y_train, ptr, y_test, pte, met.train.auc, met.test.auc, folder / "roc.png")
    plots.plot_shap_importance(imp, folder / "shap_importance.png", cfg.shap.top_n, descriptions)
    plots.plot_shap_beeswarm(s, dp, fn, folder / "shap_impact.png", cfg.shap.top_n)
    plots.plot_optuna_history(st, folder / "optuna_history.png")
    export.save_model(est, bp, imp, folder)

    del est, st, ptr, pte, s, dp, Xs
    gc.collect()

summary = pd.DataFrame(summary_rows)
consolidated = pd.DataFrame(importances)
plots.plot_comparison(summary, output_dir / "_comparison.png")
plots.plot_consolidated_shap(importances, output_dir / "_shap_consolidated.png",
                             cfg.shap.top_n, descriptions)
export.write_summary_excel(summary, consolidated, descriptions, output_dir / "_summary.xlsx")
print("\nTermine.")
display(summary.set_index("model").round(4))

In [ ]:
# --- 6b  Variante une ligne (delegue tout au module) ---
# from challenger_benchmark.pipeline import run_with_config
# summary = run_with_config(cfg)
# summary

## 7. Synthese

Recharge l'Excel recapitulatif et affiche les deux figures transverses : la comparaison des modeles et la carte de chaleur des rangs SHAP (l'argument « les risk drivers coincident entre familles »).

In [ ]:
output_dir = Path(cfg.output_dir)

perf = pd.read_excel(output_dir / "_summary.xlsx", sheet_name="performances")
display(perf.set_index("model").round(4))

show(output_dir / "_comparison.png")
show(output_dir / "_shap_consolidated.png")